In [5]:
import pandas as pd
import numpy as np
import sys
from typing import Dict, Optional, Tuple, Any, List
import math
import json
import matplotlib.pyplot as plt
import warnings
from scipy.stats import spearmanr
import seaborn as sns
from CellDecovolution_DifferentialAnalysis import differential_deconvolution_analysis
from Utils import compute_jaccard_indices, extract_significances
from Utils import spearman_across_datasets

dataset_dict = {}
original_data = pd.read_csv("../../Data/original_data.csv", index_col = 0)

responderCrit = original_data['Harmonized_Confirmed_BOR_3_Cat'].isin(['CR/PR'])
progressorCrit = original_data['Harmonized_Confirmed_BOR_3_Cat'].isin(['PD','SD'])

responders = original_data[responderCrit].index
progressors = original_data[progressorCrit].index

original_data.loc[responders,"Labels"] = 'Responder'
original_data.loc[progressors,"Labels"] = 'Progressor'

dataset_dict["Origin"] = original_data
seeds = [42,0]
for seed in seeds:
    syn_datas = [f"avatarsk5_{seed}",f"avatarsk10_{seed}",f"ctgan_{seed}",f"gaussiancopula_{seed}",f"synthpop_{seed}",f"tvae_{seed}"]
    for i, tool in enumerate(syn_datas):
        syn_df = pd.read_csv(f"../../Data/{syn_datas[i]}.csv", index_col = 0)
        responderCrit = syn_df['Harmonized_Confirmed_BOR_3_Cat'].isin(['CR/PR'])
        progressorCrit = syn_df['Harmonized_Confirmed_BOR_3_Cat'].isin(['PD','SD'])
        
        responders = syn_df[responderCrit].index
        progressors = syn_df[progressorCrit].index
        
        syn_df.loc[responders,"Labels"] = 'Responder'
        syn_df.loc[progressors,"Labels"] = 'Progressor'
        dataset_dict[tool] = syn_df
    
    dataset_names = ["Origin", f"avatarsk5_{seed}", f"avatarsk10_{seed}", 
                     f"ctgan_{seed}", f"gaussiancopula_{seed}", f"synthpop_{seed}"]
    
    results_deco_dict = {}
    
    for dataset_name in dataset_names:
        # Load deconvolution results
        deconv_result = pd.read_csv(f"ResultsCibersortx/CIBERSORTx_{dataset_name}_Results.csv")
        deconv_result = deconv_result.iloc[:,0:23]
        # Prepare metadata from dataset
        data = dataset_dict[dataset_name]
        data = data[~data['Labels'].isna()]
        
        metadata = pd.DataFrame({
            "Patient_ID": data.index.values.tolist(),
            "Labels": data["Labels"].values.tolist(),
        })
        
        # Define comparison groups
        comparison_groups = {
            'group1': ['Responder'],
            'group2': ['Progressor']
        }
        
        # Run differential analysis
        result_df = differential_deconvolution_analysis(
            deconvolution_result=deconv_result,
            metadata=metadata,
            group_column='Labels',
            comparison_groups=comparison_groups,
            cell_type_columns=None,  # Auto-detect
            sample_id_column='Mixture',
            normalize=True,
            alpha=0.05,
            correction_method='fdr_bh'
        )
        
        results_deco_dict[dataset_name] = result_df
        # Save results
        result_df.to_csv(f"ResultsDA/Seed_{seed}/{dataset_name}_differential_deconvolution.csv", index=False)
    
    orig_cells = extract_significances(results_deco_dict["Origin"], term_col="Cell Type", adj_p_col="Q_value", threshold=0.05)
    synth_cells_dict = {k: extract_significances(v, term_col="Cell Type", adj_p_col="Q_value", threshold=0.05)
                         for k, v in results_deco_dict.items() if k != "Origin"}
    
    jaccard_df = compute_jaccard_indices(orig_cells, synth_cells_dict)
    jaccard_df.to_csv(f'ResultsDA/Seed_{seed}/JaccardIndex_{seed}.csv')
    
    
    
    spearman_df,_ = spearman_across_datasets(
        gsea_overall=results_deco_dict,
        origin="Origin",
        term_col="Cell Type",
        score_col="Diff_Mean",
        min_terms=3,
    )
    spearman_df.to_csv(f'ResultsDA/Seed_{seed}/Spearman_{seed}.csv')